In [ ]:
file_path1 = os.path.expanduser("~/Downloads/Lung_atlas_public.h5ad")
adata_lung = sc.read_h5ad(file_path1)
adata_lung

In [ ]:
import os
import numpy as np
import scanpy as sc

file_path = os.path.expanduser("~/Desktop/adata_lung_qc.h5ad")
adata_lung_qc = sc.read_h5ad(file_path)

In [ ]:
def stratified_subsample_adata(adata, group_key="batch", frac=0.75, random_state=0):
    rng = np.random.default_rng(random_state)
    selected_idx = []

    obs = adata.obs.copy()

    for group, idx in obs.groupby(group_key).indices.items():
        idx = np.array(list(idx))
        n_group = len(idx)
        n_take = max(1, int(np.floor(n_group * frac)))
        chosen = rng.choice(idx, size=n_take, replace=False)
        selected_idx.extend(chosen.tolist())

    selected_idx = np.array(selected_idx)
    return adata[selected_idx].copy()

In [ ]:
adata_lung_75_rep1 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.75,
    random_state=1001
)

adata_lung_75_rep2 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.75,
    random_state=1002
)

adata_lung_75_rep3 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.75,
    random_state=1003
)


In [ ]:
adata_lung_75_rep1.write(os.path.expanduser("~/Desktop/adata_lung_75_rep1.h5ad"))
adata_lung_75_rep2.write(os.path.expanduser("~/Desktop/adata_lung_75_rep2.h5ad"))
adata_lung_75_rep3.write(os.path.expanduser("~/Desktop/adata_lung_75_rep3.h5ad"))

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import harmonypy as hm
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_harmony_75 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/adata_lung_75_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)
    adata_harmony = adata_sub.copy()
    adata_harmony.X = adata_harmony.layers["counts"].copy()

    sc.pp.normalize_total(adata_harmony, target_sum=1e4)
    sc.pp.log1p(adata_harmony)

    sc.pp.highly_variable_genes(
        adata_harmony,
        flavor="seurat",
        batch_key="batch",
        n_top_genes=2000
    )

    adata_hvg = adata_harmony[:, adata_harmony.var["highly_variable"]].copy()

    sc.tl.pca(adata_hvg)
    ho = hm.run_harmony(
        adata_hvg.obsm["X_pca"],
        adata_hvg.obs,
        vars_use=["batch"]
    )

    adata_hvg.obsm["X_pca_harmony"] = np.array(ho.Z_corr)
    sc.pp.neighbors(adata_hvg, use_rep="X_pca_harmony")
    sc.tl.leiden(adata_hvg, resolution=0.5)

    X_harmony = adata_hvg.obsm["X_pca_harmony"]

    nn_harmony = pynndescent(
        X_harmony,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_harmony,
        adata_hvg.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_harmony,
        adata_hvg.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    results_harmony_75.append({
        "method": "Harmony",
        "frac": 0.75,
        "repeat": rep,
        "n_cells": int(adata_hvg.n_obs),
        "n_hvg": int(adata_hvg.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_harmony_75[-1])
    del adata_sub, adata_harmony, adata_hvg, ho, nn_harmony, X_harmony
    gc.collect()

results_harmony_75_df = pd.DataFrame(results_harmony_75)

print("\nAll 3 repeats:")
print(results_harmony_75_df)

print("\nMean across 3 repeats:")
print(results_harmony_75_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD across 3 repeats:")
print(results_harmony_75_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_scvi_75 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/adata_lung_75_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)

    scvi.settings.seed = 1000 + rep

    scvi.model.SCVI.setup_anndata(
        adata_sub,
        layer="counts",
        batch_key="batch"
    )

    model = scvi.model.SCVI(
        adata_sub,
        n_latent=30
    )

    model.train()

    adata_sub.obsm["X_scVI"] = model.get_latent_representation()

    sc.pp.neighbors(adata_sub, use_rep="X_scVI")
    sc.tl.leiden(adata_sub, resolution=0.5)

    X_scvi = adata_sub.obsm["X_scVI"]

    nn_scvi = pynndescent(
        X_scvi,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_scvi,
        adata_sub.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_scvi,
        adata_sub.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    results_scvi_75.append({
        "method": "scVI",
        "frac": 0.75,
        "repeat": rep,
        "n_cells": int(adata_sub.n_obs),
        "n_genes": int(adata_sub.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_scvi_75[-1])

    del adata_sub, model, nn_scvi, X_scvi
    gc.collect()

results_scvi_75_df = pd.DataFrame(results_scvi_75)

print("\nAll 3 repeats:")
print(results_scvi_75_df)

print("\nMean across 3 repeats:")
print(results_scvi_75_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD across 3 repeats:")
print(results_scvi_75_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
import os
import numpy as np
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

file_path = os.path.expanduser("~/Desktop/adata_lung_75_rep3.h5ad")
adata_sub = sc.read_h5ad(file_path)

print(adata_sub)

scvi.settings.seed = 1003

scvi.model.SCVI.setup_anndata(
    adata_sub,
    layer="counts",
    batch_key="batch"
)

model = scvi.model.SCVI(
    adata_sub,
    n_latent=30
)

model.train()

adata_sub.obsm["X_scVI"] = model.get_latent_representation()

sc.pp.neighbors(adata_sub, use_rep="X_scVI")
sc.tl.leiden(adata_sub, resolution=0.5)

X_scvi = adata_sub.obsm["X_scVI"]

nn_scvi = pynndescent(
    X_scvi,
    n_neighbors=30,
    random_state=1003
)

ilisi = sm.ilisi_knn(
    nn_scvi,
    adata_sub.obs["batch"].to_numpy(),
    scale=True
)

clisi = sm.clisi_knn(
    nn_scvi,
    adata_sub.obs["cell_type"].to_numpy(),
    scale=True
)

ari = adjusted_rand_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

nmi = normalized_mutual_info_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

result_rep3 = {
    "method": "scVI",
    "frac": 0.75,
    "repeat": 3,
    "n_cells": int(adata_sub.n_obs),
    "n_genes": int(adata_sub.n_vars),
    "ilisi": float(ilisi),
    "clisi": float(clisi),
    "ari": float(ari),
    "nmi": float(nmi)
}

print("\nscVI 75% repeat 3 result:")
print(result_rep3)

In [ ]:
import os
import pandas as pd
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def compute_metrics_from_one_file(
    csv_path,
    batch_col="batch",
    celltype_col="cell_type",
    cluster_col="cluster"
):
    df = pd.read_csv(csv_path)

    embed_cols = [c for c in df.columns if c.startswith("PC_")]
    X = df[embed_cols].to_numpy()

    nn = pynndescent(
        X,
        n_neighbors=30,
        random_state=0
    )

    ilisi = sm.ilisi_knn(
        nn,
        df[batch_col].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn,
        df[celltype_col].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        df[celltype_col],
        df[cluster_col]
    )

    nmi = normalized_mutual_info_score(
        df[celltype_col],
        df[cluster_col]
    )

    return {
        "n_cells": len(df),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    }

results_seurat_75 = []
base_dir = os.path.expanduser("~/Desktop")

for rep in [1, 2, 3]:
    file_path = os.path.join(base_dir, f"lung_seurat_75_rep{rep}.csv")

    metrics = compute_metrics_from_one_file(
        csv_path=file_path,
        batch_col="batch",
        celltype_col="cell_type",
        cluster_col="cluster"
    )

    results_seurat_75.append({
        "method": "Seurat",
        "frac": 0.75,
        "repeat": rep,
        **metrics
    })

results_seurat_75_df = pd.DataFrame(results_seurat_75)

print("Repeat-level results:")
print(results_seurat_75_df)

print("\nMean:")
print(results_seurat_75_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_seurat_75_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_fastmnn_75 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/lung75_{rep}_fastmnn.h5ad")
    adata_mnn = sc.read_h5ad(file_path)

    # Use corrected embedding from fastMNN
    X_mnn = adata_mnn.obsm["corrected"]

    # Neighbors / Leiden
    sc.pp.neighbors(adata_mnn, use_rep="corrected")
    sc.tl.leiden(adata_mnn, resolution=0.5)

    # LISI metrics
    nn_mnn = pynndescent(
        X_mnn,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_mnn,
        adata_mnn.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_mnn,
        adata_mnn.obs["cell_type"].to_numpy(),
        scale=True
    )

    # ARI / NMI
    ari = adjusted_rand_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    results_fastmnn_75.append({
        "method": "fastMNN",
        "frac": 0.75,
        "repeat": rep,
        "n_cells": int(adata_mnn.n_obs),
        "n_genes": int(adata_mnn.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_fastmnn_75[-1])

    del adata_mnn, X_mnn, nn_mnn
    gc.collect()

results_fastmnn_75_df = pd.DataFrame(results_fastmnn_75)

print("\nRepeat-level results:")
print(results_fastmnn_75_df)

print("\nMean:")
print(results_fastmnn_75_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_fastmnn_75_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
file_path = os.path.expanduser("~/Desktop/adata_lung_qc.h5ad")
adata_lung_qc = sc.read_h5ad(file_path)

def stratified_subsample_adata(adata, group_key="batch", frac=0.50, random_state=0):
    rng = np.random.default_rng(random_state)
    selected_idx = []

    obs = adata.obs.copy()

    for group, idx in obs.groupby(group_key).indices.items():
        idx = np.array(list(idx))
        n_group = len(idx)
        n_take = max(1, int(np.floor(n_group * frac)))
        chosen = rng.choice(idx, size=n_take, replace=False)
        selected_idx.extend(chosen.tolist())

    selected_idx = np.array(selected_idx)
    return adata[selected_idx].copy()

adata_lung_50_rep1 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.50,
    random_state=1001
)

adata_lung_50_rep2 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.50,
    random_state=1002
)

adata_lung_50_rep3 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.50,
    random_state=1003
)

adata_lung_50_rep1.write(os.path.expanduser("~/Desktop/adata_lung_50_rep1.h5ad"))
adata_lung_50_rep2.write(os.path.expanduser("~/Desktop/adata_lung_50_rep2.h5ad"))
adata_lung_50_rep3.write(os.path.expanduser("~/Desktop/adata_lung_50_rep3.h5ad"))

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import harmonypy as hm
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_harmony_50 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/adata_lung_50_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)

    adata_harmony = adata_sub.copy()
    adata_harmony.X = adata_harmony.layers["counts"].copy()

    sc.pp.normalize_total(adata_harmony, target_sum=1e4)
    sc.pp.log1p(adata_harmony)

    sc.pp.highly_variable_genes(
        adata_harmony,
        flavor="seurat",
        batch_key="batch",
        n_top_genes=2000
    )

    adata_hvg = adata_harmony[:, adata_harmony.var["highly_variable"]].copy()

    sc.tl.pca(adata_hvg)

    ho = hm.run_harmony(
        adata_hvg.obsm["X_pca"],
        adata_hvg.obs,
        vars_use=["batch"]
    )

    adata_hvg.obsm["X_pca_harmony"] = np.array(ho.Z_corr)

    sc.pp.neighbors(adata_hvg, use_rep="X_pca_harmony")
    sc.tl.leiden(adata_hvg, resolution=0.5)

    X_harmony = adata_hvg.obsm["X_pca_harmony"]

    nn_harmony = pynndescent(
        X_harmony,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_harmony,
        adata_hvg.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_harmony,
        adata_hvg.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    results_harmony_50.append({
        "method": "Harmony",
        "frac": 0.50,
        "repeat": rep,
        "n_cells": int(adata_hvg.n_obs),
        "n_hvg": int(adata_hvg.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_harmony_50[-1])

    del adata_sub, adata_harmony, adata_hvg, ho, nn_harmony, X_harmony
    gc.collect()

results_harmony_50_df = pd.DataFrame(results_harmony_50)

print(results_harmony_50_df)

print(results_harmony_50_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print(results_harmony_50_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
def compute_metrics_from_one_file(
    csv_path,
    batch_col="batch",
    celltype_col="cell_type",
    cluster_col="cluster"
):
    df = pd.read_csv(csv_path)

    embed_cols = [c for c in df.columns if c.startswith("PC_")]
    X = df[embed_cols].to_numpy()

    nn = pynndescent(
        X,
        n_neighbors=30,
        random_state=0
    )

    ilisi = sm.ilisi_knn(
        nn,
        df[batch_col].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn,
        df[celltype_col].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        df[celltype_col],
        df[cluster_col]
    )

    nmi = normalized_mutual_info_score(
        df[celltype_col],
        df[cluster_col]
    )

    return {
        "n_cells": len(df),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    }

results_seurat_50 = []
base_dir = os.path.expanduser("~/Desktop")

for rep in [1, 2, 3]:
    file_path = os.path.join(base_dir, f"lung_seurat_50_rep{rep}.csv")

    metrics = compute_metrics_from_one_file(
        csv_path=file_path,
        batch_col="batch",
        celltype_col="cell_type",
        cluster_col="cluster"
    )

    results_seurat_50.append({
        "method": "Seurat",
        "frac": 0.50,
        "repeat": rep,
        **metrics
    })

results_seurat_50_df = pd.DataFrame(results_seurat_50)

print("Repeat-level results:")
print(results_seurat_50_df)

print("\nMean:")
print(results_seurat_50_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_seurat_50_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
results_fastmnn_50 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/lung50_{rep}_fastmnn.h5ad")
    adata_mnn = sc.read_h5ad(file_path)

    X_mnn = adata_mnn.obsm["corrected"]

    sc.pp.neighbors(adata_mnn, use_rep="corrected")
    sc.tl.leiden(adata_mnn, resolution=0.5)

    nn_mnn = pynndescent(
        X_mnn,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_mnn,
        adata_mnn.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_mnn,
        adata_mnn.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    results_fastmnn_50.append({
        "method": "fastMNN",
        "frac": 0.50,
        "repeat": rep,
        "n_cells": int(adata_mnn.n_obs),
        "n_genes": int(adata_mnn.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_fastmnn_50[-1])

    del adata_mnn, X_mnn, nn_mnn
    gc.collect()

results_fastmnn_50_df = pd.DataFrame(results_fastmnn_50)

print("\nRepeat-level results:")
print(results_fastmnn_50_df)

print("\nMean:")
print(results_fastmnn_50_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_fastmnn_50_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
file_path = os.path.expanduser("~/Desktop/adata_lung_qc.h5ad")
adata_lung_qc = sc.read_h5ad(file_path)

def stratified_subsample_adata(adata, group_key="batch", frac=0.25, random_state=0):
    rng = np.random.default_rng(random_state)
    selected_idx = []

    obs = adata.obs.copy()

    for group, idx in obs.groupby(group_key).indices.items():
        idx = np.array(list(idx))
        n_group = len(idx)
        n_take = max(1, int(np.floor(n_group * frac)))
        chosen = rng.choice(idx, size=n_take, replace=False)
        selected_idx.extend(chosen.tolist())

    selected_idx = np.array(selected_idx)
    return adata[selected_idx].copy()

adata_lung_25_rep1 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.25,
    random_state=1001
)

adata_lung_25_rep2 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.25,
    random_state=1002
)

adata_lung_25_rep3 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.25,
    random_state=1003
)

adata_lung_25_rep1.write(os.path.expanduser("~/Desktop/adata_lung_25_rep1.h5ad"))
adata_lung_25_rep2.write(os.path.expanduser("~/Desktop/adata_lung_25_rep2.h5ad"))
adata_lung_25_rep3.write(os.path.expanduser("~/Desktop/adata_lung_25_rep3.h5ad"))

In [ ]:
results_harmony_25 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/adata_lung_25_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)

    adata_harmony = adata_sub.copy()
    adata_harmony.X = adata_harmony.layers["counts"].copy()

    sc.pp.normalize_total(adata_harmony, target_sum=1e4)
    sc.pp.log1p(adata_harmony)

    sc.pp.highly_variable_genes(
        adata_harmony,
        flavor="seurat",
        batch_key="batch",
        n_top_genes=2000
    )

    adata_hvg = adata_harmony[:, adata_harmony.var["highly_variable"]].copy()

    sc.tl.pca(adata_hvg)

    ho = hm.run_harmony(
        adata_hvg.obsm["X_pca"],
        adata_hvg.obs,
        vars_use=["batch"]
    )

    adata_hvg.obsm["X_pca_harmony"] = np.array(ho.Z_corr)

    sc.pp.neighbors(adata_hvg, use_rep="X_pca_harmony")
    sc.tl.leiden(adata_hvg, resolution=0.5)

    X_harmony = adata_hvg.obsm["X_pca_harmony"]

    nn_harmony = pynndescent(
        X_harmony,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_harmony,
        adata_hvg.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_harmony,
        adata_hvg.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    results_harmony_25.append({
        "method": "Harmony",
        "frac": 0.25,
        "repeat": rep,
        "n_cells": int(adata_hvg.n_obs),
        "n_hvg": int(adata_hvg.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_harmony_25[-1])

    del adata_sub, adata_harmony, adata_hvg, ho, nn_harmony, X_harmony
    gc.collect()

results_harmony_25_df = pd.DataFrame(results_harmony_25)

print(results_harmony_25_df)

print(results_harmony_25_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print(results_harmony_25_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
def compute_metrics_from_one_file(
    csv_path,
    batch_col="batch",
    celltype_col="cell_type",
    cluster_col="cluster"
):
    df = pd.read_csv(csv_path)

    embed_cols = [c for c in df.columns if c.startswith("PC_")]
    X = df[embed_cols].to_numpy()

    nn = pynndescent(
        X,
        n_neighbors=30,
        random_state=0
    )

    ilisi = sm.ilisi_knn(
        nn,
        df[batch_col].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn,
        df[celltype_col].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        df[celltype_col],
        df[cluster_col]
    )

    nmi = normalized_mutual_info_score(
        df[celltype_col],
        df[cluster_col]
    )

    return {
        "n_cells": len(df),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    }

results_seurat_25 = []
base_dir = os.path.expanduser("~/Desktop")

for rep in [1, 2, 3]:
    file_path = os.path.join(base_dir, f"lung_seurat_25_rep{rep}.csv")

    metrics = compute_metrics_from_one_file(
        csv_path=file_path,
        batch_col="batch",
        celltype_col="cell_type",
        cluster_col="cluster"
    )

    results_seurat_25.append({
        "method": "Seurat",
        "frac": 0.25,
        "repeat": rep,
        **metrics
    })

results_seurat_25_df = pd.DataFrame(results_seurat_25)

print("Repeat-level results:")
print(results_seurat_25_df)

print("\nMean:")
print(results_seurat_25_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_seurat_25_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_fastmnn_25 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/lung25_{rep}_fastmnn.h5ad")
    adata_mnn = sc.read_h5ad(file_path)

    X_mnn = adata_mnn.obsm["corrected"]

    sc.pp.neighbors(adata_mnn, use_rep="corrected")
    sc.tl.leiden(adata_mnn, resolution=0.5)

    nn_mnn = pynndescent(
        X_mnn,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_mnn,
        adata_mnn.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_mnn,
        adata_mnn.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    results_fastmnn_25.append({
        "method": "fastMNN",
        "frac": 0.25,
        "repeat": rep,
        "n_cells": int(adata_mnn.n_obs),
        "n_genes": int(adata_mnn.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_fastmnn_25[-1])

    del adata_mnn, X_mnn, nn_mnn
    gc.collect()

results_fastmnn_25_df = pd.DataFrame(results_fastmnn_25)

print("\nRepeat-level results:")
print(results_fastmnn_25_df)

print("\nMean:")
print(results_fastmnn_25_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_fastmnn_25_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_scvi_50 = []

for rep in [1, 2, 3]:

    file_path = os.path.expanduser(f"~/Desktop/adata_lung_50_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)

    scvi.settings.seed = 1000 + rep

    scvi.model.SCVI.setup_anndata(
        adata_sub,
        layer="counts",
        batch_key="batch"
    )

    model = scvi.model.SCVI(
        adata_sub,
        n_latent=30
    )

    model.train()

    adata_sub.obsm["X_scVI"] = model.get_latent_representation()

    sc.pp.neighbors(adata_sub, use_rep="X_scVI")
    sc.tl.leiden(adata_sub, resolution=0.5)

    X_scvi = adata_sub.obsm["X_scVI"]

    nn_scvi = pynndescent(
        X_scvi,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_scvi,
        adata_sub.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_scvi,
        adata_sub.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    results_scvi_50.append({
        "method": "scVI",
        "frac": 0.50,
        "repeat": rep,
        "n_cells": int(adata_sub.n_obs),
        "n_genes": int(adata_sub.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_scvi_50[-1])

    del adata_sub, model, nn_scvi, X_scvi
    gc.collect()

results_scvi_50_df = pd.DataFrame(results_scvi_50)

print("\nAll 3 repeats:")
print(results_scvi_50_df)

print("\nMean across 3 repeats:")
print(results_scvi_50_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD across 3 repeats:")
print(results_scvi_50_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
import os
import numpy as np
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

file_path = os.path.expanduser("~/Desktop/adata_lung_50_rep3.h5ad")
adata_sub = sc.read_h5ad(file_path)

print(adata_sub)

scvi.settings.seed = 1003

scvi.model.SCVI.setup_anndata(
    adata_sub,
    layer="counts",
    batch_key="batch"
)

model = scvi.model.SCVI(
    adata_sub,
    n_latent=30
)

model.train()

adata_sub.obsm["X_scVI"] = model.get_latent_representation()

sc.pp.neighbors(adata_sub, use_rep="X_scVI")
sc.tl.leiden(adata_sub, resolution=0.5)

X_scvi = adata_sub.obsm["X_scVI"]

nn_scvi = pynndescent(
    X_scvi,
    n_neighbors=30,
    random_state=1003
)

ilisi = sm.ilisi_knn(
    nn_scvi,
    adata_sub.obs["batch"].to_numpy(),
    scale=True
)

clisi = sm.clisi_knn(
    nn_scvi,
    adata_sub.obs["cell_type"].to_numpy(),
    scale=True
)

ari = adjusted_rand_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

nmi = normalized_mutual_info_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

result_rep3 = {
    "method": "scVI",
    "frac": 0.50,
    "repeat": 3,
    "n_cells": int(adata_sub.n_obs),
    "n_genes": int(adata_sub.n_vars),
    "ilisi": float(ilisi),
    "clisi": float(clisi),
    "ari": float(ari),
    "nmi": float(nmi)
}

print("\nscVI 50% repeat 3 result:")
print(result_rep3)

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_scvi_25 = []

for rep in [1, 2, 3]:

    file_path = os.path.expanduser(f"~/Desktop/adata_lung_25_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)

    scvi.settings.seed = 1000 + rep

    scvi.model.SCVI.setup_anndata(
        adata_sub,
        layer="counts",
        batch_key="batch"
    )

    model = scvi.model.SCVI(
        adata_sub,
        n_latent=30
    )

    model.train()

    adata_sub.obsm["X_scVI"] = model.get_latent_representation()

    sc.pp.neighbors(adata_sub, use_rep="X_scVI")
    sc.tl.leiden(adata_sub, resolution=0.5)

    X_scvi = adata_sub.obsm["X_scVI"]

    nn_scvi = pynndescent(
        X_scvi,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_scvi,
        adata_sub.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_scvi,
        adata_sub.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    results_scvi_25.append({
        "method": "scVI",
        "frac": 0.25,
        "repeat": rep,
        "n_cells": int(adata_sub.n_obs),
        "n_genes": int(adata_sub.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_scvi_25[-1])

    del adata_sub, model, nn_scvi, X_scvi
    gc.collect()

results_scvi_25_df = pd.DataFrame(results_scvi_25)

print("\nAll 3 repeats:")
print(results_scvi_25_df)

print("\nMean across 3 repeats:")
print(results_scvi_25_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD across 3 repeats:")
print(results_scvi_25_df[["ilisi", "clisi", "ari", "nmi"]].std())

In [ ]:
import os
import numpy as np
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

file_path = os.path.expanduser("~/Desktop/adata_lung_25_rep2.h5ad")
adata_sub = sc.read_h5ad(file_path)

print(adata_sub)

scvi.settings.seed = 1002

scvi.model.SCVI.setup_anndata(
    adata_sub,
    layer="counts",
    batch_key="batch"
)

model = scvi.model.SCVI(
    adata_sub,
    n_latent=30
)

model.train()

adata_sub.obsm["X_scVI"] = model.get_latent_representation()

sc.pp.neighbors(adata_sub, use_rep="X_scVI")
sc.tl.leiden(adata_sub, resolution=0.5)

X_scvi = adata_sub.obsm["X_scVI"]

nn_scvi = pynndescent(
    X_scvi,
    n_neighbors=30,
    random_state=1002
)

ilisi = sm.ilisi_knn(
    nn_scvi,
    adata_sub.obs["batch"].to_numpy(),
    scale=True
)

clisi = sm.clisi_knn(
    nn_scvi,
    adata_sub.obs["cell_type"].to_numpy(),
    scale=True
)

ari = adjusted_rand_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

nmi = normalized_mutual_info_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

result_rep2 = {
    "method": "scVI",
    "frac": 0.25,
    "repeat": 2,
    "n_cells": int(adata_sub.n_obs),
    "n_genes": int(adata_sub.n_vars),
    "ilisi": float(ilisi),
    "clisi": float(clisi),
    "ari": float(ari),
    "nmi": float(nmi)
}

print("\nscVI 25% repeat 2 result:")
print(result_rep2)

In [ ]:
import os
import numpy as np
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

file_path = os.path.expanduser("~/Desktop/adata_lung_25_rep3.h5ad")
adata_sub = sc.read_h5ad(file_path)

print(adata_sub)

scvi.settings.seed = 1003

scvi.model.SCVI.setup_anndata(
    adata_sub,
    layer="counts",
    batch_key="batch"
)

model = scvi.model.SCVI(
    adata_sub,
    n_latent=30
)

model.train()

adata_sub.obsm["X_scVI"] = model.get_latent_representation()

sc.pp.neighbors(adata_sub, use_rep="X_scVI")
sc.tl.leiden(adata_sub, resolution=0.5)

X_scvi = adata_sub.obsm["X_scVI"]

nn_scvi = pynndescent(
    X_scvi,
    n_neighbors=30,
    random_state=1003
)

ilisi = sm.ilisi_knn(
    nn_scvi,
    adata_sub.obs["batch"].to_numpy(),
    scale=True
)

clisi = sm.clisi_knn(
    nn_scvi,
    adata_sub.obs["cell_type"].to_numpy(),
    scale=True
)

ari = adjusted_rand_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

nmi = normalized_mutual_info_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

result_rep3 = {
    "method": "scVI",
    "frac": 0.25,
    "repeat": 3,
    "n_cells": int(adata_sub.n_obs),
    "n_genes": int(adata_sub.n_vars),
    "ilisi": float(ilisi),
    "clisi": float(clisi),
    "ari": float(ari),
    "nmi": float(nmi)
}

print("\nscVI 25% repeat 3 result:")
print(result_rep3)